In [1]:
!pip install tensorflow transformers

In [2]:
!pip install tf-keras

In [3]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import regularizers
from transformers import BertTokenizer, TFBertForSequenceClassification
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder

In [4]:
# Upload your dataset
from google.colab import files
uploaded = files.upload()

filename = list(uploaded.keys())[0]

Saving isear_processed_dataset.csv to isear_processed_dataset.csv


In [5]:
# Load CSV dataset
df = pd.read_csv(filename)

In [6]:
# Preprocess labels (if labels are strings like "happy", "sad")
le = LabelEncoder()
df["label"] = le.fit_transform(df["emotion"])  # Convert text labels to integers
num_classes = len(le.classes_)  # Number of emotion classes


In [7]:
# Split data into train, validation, and test sets
train_df, temp_df = train_test_split(df, test_size=0.25, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

In [8]:
# Extract texts and labels
train_texts, train_labels = train_df["cleaned_text"].tolist(), train_df["label"].values
val_texts, val_labels = val_df["cleaned_text"].tolist(), val_df["label"].values
test_texts, test_labels = test_df["cleaned_text"].tolist(), test_df["label"].values

In [9]:
df

,text,emotion,cleaned_text,label
0,"During the period of falling in love, each tim...",joy,during the period of falling in love each time...,4
1,When I was involved in a traffic accident.,fear,when i was involved in a traffic accident,2
2,When I was driving home after several days of...,anger,when i was driving home after several days of ...,0
3,When I lost the person who meant the most to me.,sadness,when i lost the person who meant the most to me,5
4,The time I knocked a deer down - the sight of ...,disgust,the time i knocked a deer down the sight of th...,1
...,...,...,...,...
7508,Two years back someone invited me to be the tu...,anger,two years back someone invited me to be the tu...,0
7509,I had taken the responsibility to do something...,sadness,i had taken the responsibility to do something...,5
7510,I was at home and I heard a loud sound of spit...,disgust,i was at home and i heard a loud sound of spit...,1
7511,I did not do the homework that the teacher had...,shame,i did not do the homework that the teacher had...,6


In [10]:
# Find max sentence length per row (based on word count)
def max_sentence_length(text):
    sentences = text.split('.')  # you can use nltk.sent_tokenize() for better results
    return max(len(sentence.split()) for sentence in sentences if sentence.strip())

df['max_sentence_length'] = df['cleaned_text'].apply(max_sentence_length)

# Find the row with the maximum sentence length
max_len_row = df.loc[df['max_sentence_length'].idxmax()]

# Print results
print("Maximum sentence length (in words):", max_len_row['max_sentence_length'])
print("Emotion label:", max_len_row['emotion'])
print("Text with longest sentence:\n", max_len_row['cleaned_text'])


Maximum sentence length (in words): 179
Emotion label: disgust
Text with longest sentence:
 a few days back i was waiting for the bus at the bus stop before getting into the bus i had prepared the exact amount of coins to pay for the bus fair and when i got into the bus i put these coins into the box meant to collect the bus fair i thought that i had paid and wanted to get inside however the bus driver called me and asked me in an impolite way if the coins were stuck at the opening of the box he had not seen me paying and there was not a stack of coins in the box i could not understand this and the driver kept questioning me he made me feel angry and at last i inserted a dollar coin in the box just to get away from him later i found that i had forgotten a few coins in my pocket and had not paid enough for the fair the first time after i had entered the bus i could still hear him scolding me and i felt disgusted


In [11]:
# Initialize BERT tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
max_length = 179  # Adjust based on your text length
dropout_rate=0.25

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

In [12]:
# Tokenize all splits
def tokenize(texts):
    return tokenizer(
        texts, padding="max_length", truncation=True, max_length=max_length, return_tensors="tf"
    )

In [13]:
train_encodings = tokenize(train_texts)
val_encodings = tokenize(val_texts)
test_encodings = tokenize(test_texts)


In [14]:
# Convert labels to TensorFlow tensors
train_labels = tf.convert_to_tensor(train_labels)
val_labels = tf.convert_to_tensor(val_labels)
test_labels = tf.convert_to_tensor(test_labels)


In [15]:
# Create TF Datasets for non-BERT models
batch_size = 64

train_dataset = tf.data.Dataset.from_tensor_slices((
    {"input_ids": train_encodings["input_ids"]},
    train_labels
)).batch(batch_size).prefetch(tf.data.AUTOTUNE)

val_dataset = tf.data.Dataset.from_tensor_slices((
    {"input_ids": val_encodings["input_ids"]},
    val_labels
)).batch(batch_size).prefetch(tf.data.AUTOTUNE)

test_dataset = tf.data.Dataset.from_tensor_slices((
    {"input_ids": test_encodings["input_ids"]},
    test_labels
)).batch(batch_size).prefetch(tf.data.AUTOTUNE)


In [16]:
from tensorflow.keras import regularizers, layers
# from tensorflow.keras.regularizers import l2


def build_model(model_type):
    vocab_size = tokenizer.vocab_size
    embedding_dim = 128
    dropout_rate = 0.25
    l2_reg = 0.01

    inputs = layers.Input(shape=(max_length,), name="input_ids")
    x = layers.Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        mask_zero=True
    )(inputs)

    # Architecture Selection
    if model_type == "bilstm":
        x = layers.Bidirectional(layers.LSTM(64))(x)

    elif model_type == "bigru":
        x = layers.Bidirectional(layers.GRU(64))(x)

    elif model_type == "rnn":
        x = layers.SimpleRNN(64)(x)
    elif model_type == "cnn":
        # Multi-scale CNN
        conv3 = layers.Conv1D(64, 3, activation="relu", padding="same")(x)
        pool3 = layers.GlobalMaxPooling1D()(conv3)
        conv4 = layers.Conv1D(64, 4, activation="relu", padding="same")(x)
        pool4 = layers.GlobalMaxPooling1D()(conv4)
        conv5 = layers.Conv1D(64, 5, activation="relu", padding="same")(x)
        pool5 = layers.GlobalMaxPooling1D()(conv5)
        x = layers.concatenate([pool3, pool4, pool5])
    elif model_type == "ann":
        x = layers.GlobalAveragePooling1D()(x)
        x = layers.Dense(64, activation="relu")(x)
                        # kernel_regularizer=regularizers.l2(l2_reg))(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(dropout_rate)(x)
        # x = layers.Dense(64, activation="relu")(x)
                        # kernel_regularizer=regularizers.l2(l2_reg))(x)
    else:
        raise ValueError(f"Invalid model_type: {model_type}")

    # Common Classifier Head
    x = layers.Dropout(dropout_rate)(x)
    x = layers.Dense(64, activation="relu")(x)
                    # kernel_regularizer=regularizers.l2(l2_reg))(x)
    x = layers.Dropout(dropout_rate)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    return tf.keras.Model(inputs=inputs, outputs=outputs)

In [17]:
# import tensorflow as tf
from tensorflow.keras.optimizers import Adam
# from transformers import BertTokenizer, TFBertForSequenceClassification


In [18]:
models = {
    "BiLSTM": build_model("bilstm"),
    "BiGRU": build_model("bigru"),
    "CNN": build_model("cnn"),
    "ANN": build_model("ann"),
    "RNN": build_model("rnn"),
    # "Transformer": build_model("transformer")
}

/usr/local/lib/python3.11/dist-packages/keras/src/layers/layer.py:938: UserWarning: Layer 'conv1d' (of type Conv1D) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/keras/src/layers/layer.py:938: UserWarning: Layer 'conv1d_1' (of type Conv1D) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/keras/src/layers/layer.py:938: UserWarning: Layer 'conv1d_2' (of type Conv1D) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


In [19]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

In [20]:
callback = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)


In [21]:
from sklearn.metrics import classification_report

# Initialize results dictionary
results = {}

# Training loop
for name, model in models.items():
    print(f"\nTraining {name}...")

    # For custom models
    train_data = train_dataset
    val_data = val_dataset
    test_data = test_dataset
    optimizer = "adam"

    # Compile model

    model.compile(
        optimizer=optimizer,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    # Train with EarlyStopping
    model.fit(
        train_data,
        validation_data=val_data,# if name not in ["BERT", "GPT-2", "RoBERTa"] else None,
        epochs=10,
        callbacks=[callback],# if name not in ["BERT", "GPT-2", "RoBERTa"] else None,
        verbose=1
    )

    # Evaluation
    y_pred = model.predict(test_data)
    y_pred = np.argmax(y_pred, axis=1)

    # ... (your evaluation logic: classification_report, confusion_matrix etc.)
    # Generate classification report
    report = classification_report(test_labels, y_pred, output_dict=True)

    # Store results
    results[name] = {
        "accuracy": report["accuracy"],
        "f1": report["macro avg"]["f1-score"],
        "weighted_f1": report["weighted avg"]["f1-score"],
        "precision": report["macro avg"]["precision"],
        "recall": report["macro avg"]["recall"]
    }



Training BiLSTM...
Epoch 1/10


/usr/local/lib/python3.11/dist-packages/keras/src/models/functional.py:237: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: input_ids
Received: inputs=['Tensor(shape=(None, 179))']
  warnings.warn(msg)


89/89 ━━━━━━━━━━━━━━━━━━━━ 7s 20ms/step - accuracy: 0.1800 - loss: 1.9252 - val_accuracy: 0.3727 - val_loss: 1.6431
Epoch 2/10
89/89 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.4339 - loss: 1.4857 - val_accuracy: 0.4324 - val_loss: 1.4595
Epoch 3/10
89/89 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.6151 - loss: 1.0343 - val_accuracy: 0.4973 - val_loss: 1.3926
Epoch 4/10
89/89 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.7370 - loss: 0.7510 - val_accuracy: 0.5389 - val_loss: 1.4564
Epoch 5/10
89/89 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.8111 - loss: 0.5547 - val_accuracy: 0.5240 - val_loss: 1.6532
Epoch 6/10
89/89 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.8513 - loss: 0.4373 - val_accuracy: 0.5016 - val_loss: 1.9074
Epoch 7/10
89/89 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.8845 - loss: 0.3608 - val_accuracy: 0.5325 - val_loss: 1.7927
Epoch 8/10
89/89 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.9109 - loss: 0.2794 - val_accuracy: 0.5421 - val_loss: 1.

/usr/local/lib/python3.11/dist-packages/keras/src/models/functional.py:237: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: input_ids
Received: inputs=['Tensor(shape=(64, 179))']
  warnings.warn(msg)


15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step

Training BiGRU...
Epoch 1/10


...

15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step

Training RNN...
Epoch 1/10


/usr/local/lib/python3.11/dist-packages/keras/src/models/functional.py:237: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: input_ids
Received: inputs=['Tensor(shape=(None, 179))']
  warnings.warn(msg)


89/89 ━━━━━━━━━━━━━━━━━━━━ 11s 61ms/step - accuracy: 0.1480 - loss: 1.9520 - val_accuracy: 0.1800 - val_loss: 1.9318
Epoch 2/10
89/89 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.3253 - loss: 1.8269 - val_accuracy: 0.2492 - val_loss: 1.8240
Epoch 3/10
89/89 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.4839 - loss: 1.4678 - val_accuracy: 0.3493 - val_loss: 1.8383
Epoch 4/10
89/89 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.6939 - loss: 0.8803 - val_accuracy: 0.3248 - val_loss: 2.0523
Epoch 5/10
89/89 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.8546 - loss: 0.4534 - val_accuracy: 0.3291 - val_loss: 2.3645
Epoch 6/10
89/89 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.9218 - loss: 0.2616 - val_accuracy: 0.3365 - val_loss: 2.6475
Epoch 7/10
89/89 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.9582 - loss: 0.1499 - val_accuracy: 0.3206 - val_loss: 2.8882
Epoch 7: early stopping
Restoring model weights from the end of the best epoch: 2.


/usr/local/lib/python3.11/dist-packages/keras/src/models/functional.py:237: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: input_ids
Received: inputs=['Tensor(shape=(64, 179))']
  warnings.warn(msg)


15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 45ms/step


In [22]:
# Results comparison
results_df = pd.DataFrame(results).T
print("\nModel Comparison:")
print(results_df.sort_values(by="f1", ascending=False))


Model Comparison:
        accuracy        f1  weighted_f1  precision    recall
CNN     0.609574  0.608716     0.612843   0.614856  0.606662
ANN     0.539362  0.541269     0.545196   0.554600  0.538088
BiLSTM  0.531915  0.536182     0.540368   0.563663  0.529907
BiGRU   0.493617  0.506362     0.512437   0.557860  0.491137
RNN     0.267021  0.241252     0.245165   0.256591  0.260179
